# 01. Feature engineering en Databricks

Este notebook lee el CSV raw, genera las features sin leakage y guarda el dataset listo para modelado.

In [3]:
import sys
from pathlib import Path

def _add_databricks_src_path() -> None:
    candidates = []
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        notebook_root = Path(notebook_path).parent.parent
        candidates.extend([notebook_root / "src", notebook_root.parent / "src"])
    except Exception:
        pass

    cwd = Path.cwd()
    candidates.extend([cwd / "src", cwd / "databricks" / "src", cwd.parent / "databricks" / "src"])

    for candidate in candidates:
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))

_add_databricks_src_path()

try:
    from fraudshield_databricks import build_feature_engineered_dataset_spark
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        "No se pudo importar fraudshield_databricks. Verifica que databricks/src esté sincronizado en el repo de Databricks o que el notebook se ejecute dentro del workspace correcto."
    ) from error

RAW_DATA_PATH = "/Volumes/ml/fraudshield/data"
OUTPUT_PATH = "/Volumes/ml/fraudshield/data/credit_card_transactions_fe.parquet"

ModuleNotFoundError: No se pudo importar fraudshield_databricks. Verifica que databricks/src esté sincronizado en el repo de Databricks o que el notebook se ejecute dentro del workspace correcto.

In [ ]:
df_raw = spark.read.option("header", True).option("inferSchema", True).csv(RAW_DATA_PATH)
df_fe = build_feature_engineered_dataset_spark(df_raw)

df_fe.write.mode("overwrite").parquet(OUTPUT_PATH)

print(f"Filas: {df_fe.count():,}")
print(f"Columnas totales: {len(df_fe.columns):,}")
print(f"Archivo guardado en: {OUTPUT_PATH}")

Siguiente paso: ejecutar el notebook de entrenamiento y revisar los runs en MLflow.